# Tutorial 8 - Numerical Differentiation & Integration

---

## Part A: Numerical Differentiation

## 1. Finite Difference Formulas

When we cannot compute $f'(x)$ analytically (e.g., $f$ is given as data, or is very complex), we approximate it from function values.

All formulas follow from **Taylor expansion**:
$$f(x+h) = f(x) + h f'(x) + \frac{h^2}{2}f''(x) + \frac{h^3}{6}f'''(x) + \ldots$$

### Forward Difference
$$f'(x) \approx \frac{f(x+h) - f(x)}{h} \quad \text{(1st order: error } O(h)\text{)}$$

### Backward Difference
$$f'(x) \approx \frac{f(x) - f(x-h)}{h} \quad \text{(1st order: error } O(h)\text{)}$$

### Central Difference (2nd order)
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \quad \text{(2nd order: error } O(h^2)\text{)}$$

### Central Difference (4th order)
$$f'(x) \approx \frac{-f(x+2h) + 8f(x+h) - 8f(x-h) + f(x-2h)}{12h} \quad \text{(4th order: error } O(h^4)\text{)}$$

### Second Derivative
$$f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2} \quad \text{(2nd order: error } O(h^2)\text{)}$$

In [1]:
import sys, math
sys.path.insert(0, '../')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from numethods.differentiation import (
    ForwardDiff, BackwardDiff, CentralDiff, CentralDiff4th,
    SecondDerivative, RichardsonExtrap
)

# Test function: f(x) = sin(x) * exp(-x/5)
# Exact: f'(x) = cos(x)*exp(-x/5) - sin(x)*exp(-x/5)/5
# Exact: f''(x) = ...
def f(x):
    return math.sin(x) * math.exp(-x/5)

def df_exact(x):
    return (math.cos(x) - math.sin(x)/5) * math.exp(-x/5)

def d2f_exact(x):
    return ((-math.sin(x) - math.cos(x)/5) - (math.cos(x)/5 - math.sin(x)/25)) * math.exp(-x/5)

x0 = 1.2
h = 1e-4

print(f"Test: f(x) = sin(x)·exp(-x/5) at x = {x0}")
print(f"Exact f'({x0}) = {df_exact(x0):.10f}")
print()
print(f"{'Method':<28} | {'Approx':>14} | {'Error':>12}")
print("-" * 62)

methods_diff = [
    ("Forward  (O(h))", ForwardDiff),
    ("Backward (O(h))", BackwardDiff),
    ("Central  (O(h²))", CentralDiff),
    ("Central4th(O(h⁴))", CentralDiff4th),
    ("Richardson(O(h⁴))", RichardsonExtrap),
]

for name, func in methods_diff:
    approx = func(f, x0, h)
    err = abs(approx - df_exact(x0))
    print(f"{name:<28} | {approx:14.10f} | {err:12.3e}")

print(f"\nSecond derivative:")
d2 = SecondDerivative(f, x0, h)
d2_ex = d2f_exact(x0)
print(f"  SecondDerivative = {d2:.10f}, exact = {d2_ex:.10f}, error = {abs(d2-d2_ex):.3e}")

Test: f(x) = sin(x)·exp(-x/5) at x = 1.2
Exact f'(1.2) = 0.1384071228

Method                       |         Approx |        Error
--------------------------------------------------------------
Forward  (O(h))              |   0.1383662303 |    4.089e-05
Backward (O(h))              |   0.1384480160 |    4.089e-05
Central  (O(h²))             |   0.1384071231 |    3.059e-10
Central4th(O(h⁴))            |   0.1384071228 |    8.235e-13
Richardson(O(h⁴))            |   0.1384071228 |    4.718e-13

Second derivative:
  SecondDerivative = -0.8178574706, exact = -0.8178574783, error = 7.621e-09


## 2. Effect of Step Size $h$: Truncation vs Rounding Error

There's a fundamental trade-off in choosing $h$:

- **Large $h$**: Large **truncation error** (Taylor terms neglected)
- **Small $h$**: Large **rounding error** (catastrophic cancellation in $f(x+h) - f(x)$)

The optimal $h$ balances these two effects:

| Method | Optimal $h$ | Min. achievable error |
|--------|-------------|----------------------|
| Forward diff | $h \sim \sqrt{\varepsilon_M}$ | $\sim \sqrt{\varepsilon_M}$ |
| Central diff | $h \sim \varepsilon_M^{1/3}$ | $\sim \varepsilon_M^{2/3}$ |
| 4th-order central | $h \sim \varepsilon_M^{1/5}$ | $\sim \varepsilon_M^{4/5}$ |

where $\varepsilon_M \approx 2.2 \times 10^{-16}$ is machine epsilon.

In [3]:
# h-study: error as function of h
import math

x0 = 1.2
hs = [10**(-k/2) for k in range(1, 25)]  # h from 0.3 to 1e-12

err_fwd = []
err_cen = []
err_c4  = []
err_rich = []

exact = df_exact(x0)
for h in hs:
    err_fwd.append(abs(ForwardDiff(f, x0, h) - exact))
    err_cen.append(abs(CentralDiff(f, x0, h) - exact))
    err_c4.append(abs(CentralDiff4th(f, x0, h) - exact))
    err_rich.append(abs(RichardsonExtrap(f, x0, h) - exact))

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(hs, err_fwd, 'b-o', markersize=4, linewidth=1.2, label='Forward O(h)')
ax.loglog(hs, err_cen, 'r-s', markersize=4, linewidth=1.2, label='Central O(h²)')
ax.loglog(hs, err_c4,  'g-^', markersize=4, linewidth=1.2, label='Central4th O(h⁴)')
ax.loglog(hs, err_rich, 'm-d', markersize=4, linewidth=1.2, label='Richardson O(h⁴)')

# Reference slopes
hmin, hmax = 1e-12, 0.3
ax.loglog([hmin, hmax], [hmin, hmax], 'k--', alpha=0.4, label='O(h)')
ax.loglog([hmin, hmax], [hmin**2, hmax**2], 'k:', alpha=0.4, label='O(h²)')
ax.loglog([hmin, hmax], [hmin**4, hmax**4], 'k-.', alpha=0.4, label='O(h⁴)')

ax.set_xlabel('Step size h')
ax.set_ylabel('|Error|')
ax.set_title("Finite Difference Error vs h\n(Truncation error ↘ for small h; rounding error ↗)")
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('fd_error_vs_h.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved fd_error_vs_h.png")
print("Note: each curve has a minimum; this is the optimal h for that method!")

Saved fd_error_vs_h.png
Note: each curve has a minimum; this is the optimal h for that method!


![alt text](fd_error_vs_h.png)

## 3. Richardson Extrapolation

Richardson extrapolation **eliminates the leading error term** by combining two estimates with step sizes $h$ and $h/2$.

For central differences with error $\approx c h^2 + O(h^4)$:

$$D(h) = \frac{f(x+h)-f(x-h)}{2h} = f'(x) + c h^2 + O(h^4)$$

$$D(h/2) = f'(x) + c \frac{h^2}{4} + O(h^4)$$

Combine to eliminate the $h^2$ term:
$$\frac{4 D(h/2) - D(h)}{3} = f'(x) + O(h^4) \quad \text{(4th order!)}$$

This is exactly what `RichardsonExtrap` implements. The same idea extends to arbitrarily high orders (Romberg integration).

---

## Part B: Numerical Integration (Quadrature)

## 4. The Integration Problem

Given $f: [a, b] \to \mathbb{R}$, compute:
$$I = \int_a^b f(x) \, dx$$

Analytically: possible only for simple $f$. Numerically: evaluate $f$ at $n$ points $x_0, \ldots, x_{n-1}$ and combine:
$$I \approx \sum_{i=0}^{n-1} w_i f(x_i)$$

Quadrature rules differ in **node placement** $x_i$ and **weights** $w_i$.

In [4]:
from numethods.quadrature import Trapezoidal, Simpson, GaussLegendre

# Test function: integral of sin(x) from 0 to pi = 2.0 (exact)
import math

f_int = math.sin
a, b = 0.0, math.pi
exact_I = 2.0

print("Integrating sin(x) from 0 to π (exact = 2.0)")
print()
print(f"{'Method':<32} | {'n':>6} | {'Result':>14} | {'Error':>12}")
print("-" * 72)

for n in [4, 10, 50, 100]:
    for name, cls, kwargs in [
        ("Trapezoidal", Trapezoidal, {}),
        ("Simpson", Simpson, {"n": n if n % 2 == 0 else n+1}),
    ]:
        try:
            result = cls(f_int, a, b, n).integrate()
            err = abs(result - exact_I)
            print(f"{name:<32} | {n:>6} | {result:14.10f} | {err:12.3e}")
        except Exception as e:
            print(f"{name:<32} | {n:>6} | ERROR: {e}")

# Gauss-Legendre (different n meaning: n = number of quadrature points)
for n_gl in [2, 3]:
    result = GaussLegendre(f_int, a, b, n_gl).integrate()
    err = abs(result - exact_I)
    print(f"{'Gauss-Legendre':<32} | {n_gl:>6} | {result:14.10f} | {err:12.3e}")

Integrating sin(x) from 0 to π (exact = 2.0)

Method                           |      n |         Result |        Error
------------------------------------------------------------------------
Trapezoidal                      |      4 |   1.8961188979 |    1.039e-01
Simpson                          |      4 |   2.0045597550 |    4.560e-03
Trapezoidal                      |     10 |   1.9835235375 |    1.648e-02
Simpson                          |     10 |   2.0001095173 |    1.095e-04
Trapezoidal                      |     50 |   1.9993419831 |    6.580e-04
Simpson                          |     50 |   2.0000001733 |    1.733e-07
Trapezoidal                      |    100 |   1.9998355039 |    1.645e-04
Simpson                          |    100 |   2.0000000108 |    1.082e-08
Gauss-Legendre                   |      2 |   1.9358195747 |    6.418e-02
Gauss-Legendre                   |      3 |   2.0013889136 |    1.389e-03


## 5. Composite Trapezoidal Rule

Divide $[a, b]$ into $n$ equal intervals of width $h = (b-a)/n$ with nodes $x_i = a + ih$:

$$T_n = h\left[\frac{f(x_0)}{2} + f(x_1) + f(x_2) + \cdots + f(x_{n-1}) + \frac{f(x_n)}{2}\right]$$

**Error:** $E = -\frac{(b-a)h^2}{12} f''(\xi)$ for some $\xi \in (a,b)$ → **2nd order**: error $O(h^2)$.

Doubling $n$ (halving $h$) → **4× reduction** in error.

## 6. Composite Simpson's Rule

Requires **even** $n$; uses parabolic interpolation on each pair of intervals:

$$S_n = \frac{h}{3}\left[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \cdots + 4f(x_{n-1}) + f(x_n)\right]$$

**Error:** $O(h^4)$ - **4th order**! Doubling $n$ → **16× reduction** in error.

The alternating $4,2,4,2,\ldots$ coefficients come from Simpson's $1/3$ rule applied to pairs of intervals.

In [5]:
# Convergence order verification
print("Convergence order verification for sin(x) over [0, π]")
print()

for name, cls, kwargs_fn in [
    ("Trapezoidal", Trapezoidal, lambda n: {}),
    ("Simpson", Simpson, lambda n: {}),
]:
    print(f"--- {name} ---")
    print(f"{'n':>8} | {'Error':>14} | {'Ratio':>10}")
    print("-" * 38)
    prev_err = None
    for n in [4, 8, 16, 32, 64, 128, 256]:
        if name == "Simpson" and n % 2 != 0:
            continue
        result = cls(f_int, a, b, n).integrate()
        err = abs(result - exact_I)
        if err < 1e-15:
            ratio = "—"
        elif prev_err:
            ratio = f"{prev_err/err:.2f}x"
        else:
            ratio = "—"
        print(f"{n:8d} | {err:14.3e} | {ratio:>10}")
        prev_err = err
    print()

print("Trapezoidal: ratio ≈ 4x per doubling → O(h²)")
print("Simpson:     ratio ≈ 16x per doubling → O(h⁴)")

Convergence order verification for sin(x) over [0, π]

--- Trapezoidal ---
       n |          Error |      Ratio
--------------------------------------
       4 |      1.039e-01 |          —
       8 |      2.577e-02 |      4.03x
      16 |      6.430e-03 |      4.01x
      32 |      1.607e-03 |      4.00x
      64 |      4.016e-04 |      4.00x
     128 |      1.004e-04 |      4.00x
     256 |      2.510e-05 |      4.00x

--- Simpson ---
       n |          Error |      Ratio
--------------------------------------
       4 |      4.560e-03 |          —
       8 |      2.692e-04 |     16.94x
      16 |      1.659e-05 |     16.22x
      32 |      1.033e-06 |     16.06x
      64 |      6.453e-08 |     16.01x
     128 |      4.032e-09 |     16.00x
     256 |      2.520e-10 |     16.00x

Trapezoidal: ratio ≈ 4x per doubling → O(h²)
Simpson:     ratio ≈ 16x per doubling → O(h⁴)


## 7. Gauss-Legendre Quadrature

Instead of equally-spaced nodes, choose **optimal** node positions and weights.

For an $n$-point Gauss–Legendre rule on $[-1, 1]$:
$$\int_{-1}^{1} f(x) \, dx \approx \sum_{i=1}^{n} w_i f(x_i)$$

The nodes $x_i$ are roots of the $n$-th Legendre polynomial, and weights $w_i$ are chosen to integrate **exactly all polynomials of degree $\leq 2n-1$**.

### Key property
An $n$-point Gauss–Legendre rule is **exact for polynomials of degree $\leq 2n-1$**, remarkable compared to Simpson's rule (exact up to degree 3) with 3 points.

| Method | Points | Exact for degree | Error |
|--------|--------|------------------|-------|
| Trapezoidal | $n+1$ | 1 | $O(h^2)$ |
| Simpson | $n+1$ (even $n$) | 3 | $O(h^4)$ |
| Gauss-Legendre (2-pt) | 2 | 3 | exact for $\leq$ degree 3 |
| Gauss-Legendre (3-pt) | 3 | 5 | exact for $\leq$ degree 5 |

In [6]:
# Exact integration of polynomials with 2-point Gauss-Legendre
print("Gauss-Legendre: exact for polynomials of degree ≤ 2n-1")
print()

test_cases = [
    ("1", lambda x: 1.0, 0, 1, 1.0),
    ("x", lambda x: x, 0, 1, 0.5),
    ("x²", lambda x: x**2, 0, 1, 1/3),
    ("x³", lambda x: x**3, 0, 1, 0.25),
    ("x⁴", lambda x: x**4, 0, 1, 0.2),
    ("x⁵", lambda x: x**5, 0, 1, 1/6),
    ("x⁶", lambda x: x**6, 0, 1, 1/7),
    ("sin(x)", math.sin, 0, math.pi, 2.0),
    ("e^x", math.exp, 0, 1, math.e - 1),
]

print(f"{'Integrand':<12} | {'GL-2pt':>12} | {'GL-3pt':>12} | {'Exact':>12} | {'GL2 err':>10}")
print("-" * 70)
for fname, func, a_, b_, exact_ in test_cases:
    gl2 = GaussLegendre(func, a_, b_, n=2).integrate()
    gl3 = GaussLegendre(func, a_, b_, n=3).integrate()
    print(f"{fname:<12} | {gl2:12.8f} | {gl3:12.8f} | {exact_:12.8f} | {abs(gl2-exact_):10.3e}")

Gauss-Legendre: exact for polynomials of degree ≤ 2n-1

Integrand    |       GL-2pt |       GL-3pt |        Exact |    GL2 err
----------------------------------------------------------------------
1            |   1.00000000 |   1.00000000 |   1.00000000 |  0.000e+00
x            |   0.50000000 |   0.50000000 |   0.50000000 |  0.000e+00
x²           |   0.33333333 |   0.33333333 |   0.33333333 |  0.000e+00
x³           |   0.25000000 |   0.25000000 |   0.25000000 |  2.776e-17
x⁴           |   0.19444444 |   0.20000000 |   0.20000000 |  5.556e-03
x⁵           |   0.15277778 |   0.16666667 |   0.16666667 |  1.389e-02
x⁶           |   0.12037037 |   0.14250000 |   0.14285714 |  2.249e-02
sin(x)       |   1.93581957 |   2.00138891 |   2.00000000 |  6.418e-02
e^x          |   1.71789638 |   1.71828100 |   1.71828183 |  3.855e-04


## 8. Application: Computing Geophysical Potentials

In gravity surveying, the **vertical component of gravitational attraction** of a buried 2D body along a profile is:

$$g_z(x) = 2G\rho \int_{\text{body}} \frac{(z - z')}{(x-x')^2 + (z-z')^2} dz'$$

For a simple horizontal cylinder at depth $z_0$ and radius $R$, we can integrate numerically.

In [7]:
import math

G = 6.674e-11   # gravitational constant (m³/kg/s²)
rho = 2700.0    # density contrast (kg/m³)  
z0 = 50.0       # depth to center (m)
R = 10.0        # radius (m)

# g_z at surface due to horizontal cylinder (per unit length)
# = 2 G rho pi R^2 * z0 / (x^2 + z0^2)   (exact formula for cylinder)
def gz_exact(x):
    return 2 * G * rho * math.pi * R**2 * z0 / (x**2 + z0**2)

# Numerical: integrate the cross-section
# Thin-strip model: divide cross-section into N horizontal strips at depth z',
# each of width dz', from z0-R to z0+R, with half-width sqrt(R²-(z'-z0)²)
def gz_numerical(x_obs, n_strips=50):
    z_min, z_max = z0 - R, z0 + R
    dz = (z_max - z_min) / n_strips
    total = 0.0
    for k in range(n_strips):
        z_prime = z_min + (k + 0.5) * dz  # midpoint of strip
        half_width = math.sqrt(max(R**2 - (z_prime - z0)**2, 0.0))
        strip_mass_per_len = 2 * rho * half_width * dz   # kg/m (2D body)
        total += 2 * G * strip_mass_per_len * z_prime / (x_obs**2 + z_prime**2)
    return total

# Compare along a profile
xs = [i*5.0 for i in range(-20, 21)]  # -100 to 100 m

gz_ex = [gz_exact(x) * 1e8 for x in xs]   # in mGal (* 1e8 converts to mGal-like units)
gz_num = [gz_numerical(x, n_strips=100) * 1e8 for x in xs]

max_diff = max(abs(a-b) for a,b in zip(gz_ex, gz_num))
print(f"Max difference (exact vs numerical): {max_diff:.3e} (arbitrary units)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

ax1.plot(xs, gz_ex, 'k-', linewidth=2, label='Exact (cylinder formula)')
ax1.plot(xs, gz_num, 'r--', linewidth=1.5, label='Numerical (strip integration)')
ax1.set_xlabel('Distance (m)')
ax1.set_ylabel('$g_z$ (scaled units)')
ax1.set_title(f'Gravity anomaly over horizontal cylinder (z={z0}m, R={R}m)')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.invert_yaxis()

diff_pct = [100*abs(a-b)/abs(a) if abs(a) > 1e-30 else 0 for a,b in zip(gz_ex, gz_num)]
ax2.plot(xs, diff_pct, 'b-', linewidth=1.5)
ax2.set_xlabel('Distance (m)')
ax2.set_ylabel('Relative error (%)')
ax2.set_title('Numerical integration error')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('gravity_anomaly.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved gravity_anomaly.png")

Max difference (exact vs numerical): 2.384e+00 (arbitrary units)
Saved gravity_anomaly.png


![alt text](gravity_anomaly.png)

## 9. Romberg Integration (Richardson on Trapezoidal)

By applying Richardson extrapolation **repeatedly** to the trapezoidal rule, we get **Romberg integration**:

$$T(h) = \frac{I + c_1 h^2 + c_2 h^4 + \ldots}{\text{(trapezoidal error expansion)}}$$

$$R(n,m) = \frac{4^m R(n,m-1) - R(n-1,m-1)}{4^m - 1}$$

Each column of the Romberg table increases the order by 2. Let's implement this manually to understand the concept.

In [8]:
def romberg(f, a, b, max_order=6):
    """Romberg integration: Richardson extrapolation on trapezoidal rule."""
    R = [[0.0]*(max_order+1) for _ in range(max_order+1)]
    
    # First column: trapezoidal with 2^n subintervals
    for n in range(max_order+1):
        k = 2**n  # number of intervals
        h = (b - a) / k
        s = 0.5 * (f(a) + f(b))
        for i in range(1, k):
            s += f(a + i*h)
        R[n][0] = h * s
    
    # Extrapolation columns
    for m in range(1, max_order+1):
        for n in range(m, max_order+1):
            R[n][m] = (4**m * R[n][m-1] - R[n-1][m-1]) / (4**m - 1)
    
    return R

R = romberg(math.sin, 0, math.pi, max_order=5)

print("Romberg Integration Table for ∫₀^π sin(x) dx (exact = 2)")
print()
print(f"{'n\\m':<6}", end="")
for m in range(6):
    print(f"  {'m='+str(m):>14}", end="")
print()
print("-" * 90)

for n in range(6):
    print(f"n={n:<4}", end="")
    for m in range(n+1):
        val = R[n][m]
        err = abs(val - 2.0)
        print(f"  {val:>10.8f}", end="")
    print()

print(f"\nFinal estimate: {R[5][5]:.15f}  (error: {abs(R[5][5]-2):.3e})")

Romberg Integration Table for ∫₀^π sin(x) dx (exact = 2)

n\m                m=0             m=1             m=2             m=3             m=4             m=5
------------------------------------------------------------------------------------------
n=0     0.00000000
n=1     1.57079633  2.09439510
n=2     1.89611890  2.00455975  1.99857073
n=3     1.97423160  2.00026917  1.99998313  2.00000555
n=4     1.99357034  2.00001659  1.99999975  2.00000002  1.99999999
n=5     1.99839336  2.00000103  2.00000000  2.00000000  2.00000000  2.00000000

Final estimate: 2.000000000001320  (error: 1.320e-12)


## 10. Summary

### Numerical Differentiation

| Method | Formula | Order | Optimal $h$ |
|--------|---------|-------|-------------|
| `ForwardDiff` | $(f(x+h)-f(x))/h$ | $O(h)$ | $\sqrt{\varepsilon_M} \approx 10^{-8}$ |
| `BackwardDiff` | $(f(x)-f(x-h))/h$ | $O(h)$ | $\sqrt{\varepsilon_M}$ |
| `CentralDiff` | $(f(x+h)-f(x-h))/(2h)$ | $O(h^2)$ | $\varepsilon_M^{1/3} \approx 10^{-5}$ |
| `CentralDiff4th` | 5-point stencil | $O(h^4)$ | $\varepsilon_M^{1/5} \approx 10^{-3}$ |
| `SecondDerivative` | $(f(x+h)-2f+f(x-h))/h^2$ | $O(h^2)$ | $\varepsilon_M^{1/4}$ |
| `RichardsonExtrap` | combines $h$ and $h/2$ | $O(h^4)$ | — |

### Numerical Integration

| Method | Order | When to use |
|--------|-------|-------------|
| `Trapezoidal` | $O(h^2)$ | Simple, robust, discontinuous $f'$ |
| `Simpson` | $O(h^4)$ | Smooth $f$, even number of panels |
| `GaussLegendre` | Exact for deg $\leq 2n-1$ | Small $n$, very smooth $f$, transformed integrals |

### Exercises
1. Compute the derivative of $|x|$ at $x=0$ using all methods. What happens and why?
2. For $\int_0^1 e^{-x^2}dx$ (no closed form), compare Trapezoidal, Simpson, Gauss–Legendre with 2 and 3 points.
3. Implement composite Gauss–Legendre (apply 2-point rule to $n$ subintervals). What order does it achieve?
4. Use Richardson extrapolation to estimate $f'(x_0)$ when you only have data at $x_0 - h, x_0, x_0 + h, x_0 + 2h$ (one-sided).
5. For the gravity example, experiment with different cylinder depths and radii. How does the anomaly width relate to depth?